# G10 — Notebook 02 : Analyse des résultats P02

**Problématique** : Comment le weight decay et le dropout affectent-ils la généralisation de CamemBERT sur Allociné ?  
**Ce notebook** : Chargement des résultats Optuna + Visualisations

> ⚠️ Lancer ce notebook **après** `poetry run python -m src.optimization --mode both`

---

In [ ]:
import sys, pickle, json
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from src.config import RESULTS_DIR, FIGURES_DIR
from src.visualization import (
    plot_generalization_heatmap,
    plot_convergence_curves,
    plot_optuna_importance,
    plot_sharpness_vs_f1,
)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Imports OK')

## 1. Chargement des résultats Optuna

In [ ]:
# Charger l'étude Optuna
study_path = RESULTS_DIR / 'optuna_study.pkl'
with open(study_path, 'rb') as f:
    study = pickle.load(f)

print(f'Trials complétés : {len(study.trials)}')
print(f'Meilleur F1 val  : {study.best_value:.4f}')
print(f'Meilleurs params : {study.best_params}')

In [ ]:
# DataFrame des trials
df_trials = study.trials_dataframe()
df_trials = df_trials[df_trials['state'] == 'COMPLETE'].sort_values('value', ascending=False)
df_trials.head(10)

## 2. Heatmap Gap de Généralisation × Weight Decay × Dropout

In [ ]:
# Charger les résultats de la grille déterministe
grid_path = RESULTS_DIR / 'grid_p02_results.csv'
df_grid = pd.read_csv(grid_path)
print(df_grid.to_string(index=False))

In [ ]:
plot_generalization_heatmap(df_grid, save_path=FIGURES_DIR)

## 3. Importance des hyperparamètres (Optuna)

In [ ]:
plot_optuna_importance(study, save_path=FIGURES_DIR)

## 4. Convergence — Effet du Dropout

In [ ]:
# Charger les historiques sauvegardés (un par configuration)
import json

histories = {}
for dp in [0.0, 0.1, 0.3]:
    p = RESULTS_DIR / f'history_dp{dp:.1f}.json'
    if p.exists():
        with open(p) as f:
            histories[f'dropout={dp}'] = json.load(f)

if histories:
    plot_convergence_curves(histories, save_path=FIGURES_DIR)
else:
    print('Aucun historique trouvé. Lancez l\'optimisation en premier.')

## 5. Analyse des Minima — Sharpness vs F1

In [ ]:
# Charger les données de sharpness si disponibles
sharpness_path = RESULTS_DIR / 'sharpness_results.json'
if sharpness_path.exists():
    with open(sharpness_path) as f:
        sharpness_data = json.load(f)
    plot_sharpness_vs_f1(sharpness_data, save_path=FIGURES_DIR)
else:
    print('Données de sharpness non trouvées. Lancez visualization.py en premier.')

## 6. Tableau de synthèse — Top configurations

In [ ]:
# Top configurations par F1-val
top = df_grid.sort_values('val_f1', ascending=False).head(5)

print('\n=== TOP 5 CONFIGURATIONS (P02) ===')
print(f'{"weight_decay":>15} {"dropout":>10} {"F1_val":>10} {"F1_train":>10} {"Gap":>8}')
print('-' * 58)
for _, row in top.iterrows():
    print(f'{row["weight_decay"]:>15.0e} {row["dropout"]:>10.2f} '
          f'{row["val_f1"]:>10.4f} {row["train_f1"]:>10.4f} {row["gap"]:>8.4f}')

best = top.iloc[0]
print(f'\n→ Meilleure config : weight_decay={best["weight_decay"]:.0e}, dropout={best["dropout"]:.2f}')
print(f'  F1-score val : {best["val_f1"]:.4f} | Gap : {best["gap"]:.4f}')

## 7. Conclusions P02

Complétez cette section avec vos observations :

### Effet du weight decay
- ...

### Effet du dropout
- ...

### Interaction weight_decay × dropout
- ...

### Relation flatness des minima ↔ généralisation
- ...